# Baseline Linear Regression

training linear regression in blocks using baseline features + traficom features

## 1. Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load data

In [2]:
train_path = "../../datasets/splits/train_grouped.csv"
validation_path = "../../datasets/splits/validation_grouped.csv"

In [3]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

## 3. Quick data checks

In [4]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (7997, 78)
Validation shape: (1695, 78)


In [5]:
print("Train info")
train_df.info()

print("\nValidation info")
validation_df.info()

Train info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7997 entries, 0 to 7996
Data columns (total 78 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   product_id                      7997 non-null   int64  
 1   part_name                       7997 non-null   object 
 2   price                           7997 non-null   float64
 3   quality_grade                   7997 non-null   object 
 4   oem_number                      7997 non-null   object 
 5   mileage                         7228 non-null   float64
 6   brand                           7997 non-null   object 
 7   model                           7997 non-null   object 
 8   category                        7997 non-null   object 
 9   subcategory                     7997 non-null   object 
 10  scrape_date                     7997 non-null   object 
 11  year_start                      7997 non-null   int64  
 12  year_end               

## 4. Define feature groups

Registry lifecycle features describe vehicle registration-history patterns from Traficom data. Listing dynamics features describe scrape-window behavior of listings.

In [6]:
baseline_features = [
    "part_name",
    "quality_grade",
    "oem_number",
    "mileage",
    "brand",
    "model",
    "category",
    "subcategory",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "repair_status",
]

traficom_features = [
    "model_total_registered",
    "model_median_vehicle_age",
    "model_mean_vehicle_age",
    "model_median_mileage",
    "model_mean_mileage",
    "model_median_engine_cc",
    "model_median_power_kw",
    "model_median_mass_kg",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_mean_vehicle_age",
    "brand_median_mileage",
    "brand_mean_mileage",
]

# Registry lifecycle features describe vehicle registration-history features.
registry_lifecycle_candidates = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates if column in train_df.columns
]

missing_registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates if column not in train_df.columns
]

traficom_extended_candidates = [
    "model_share_of_market",
    "model_share_within_brand",
    "model_share_over_10y",
    "model_share_over_200k_km",
    "model_automatic_share",
    "model_petrol_share",
    "model_diesel_share",
    "model_ev_share",
    "model_hybrid_share",
    "model_turbo_share",
    "brand_median_engine_cc",
    "brand_median_power_kw",
    "brand_median_mass_kg",
    "brand_share_of_market",
    "brand_share_over_10y",
    "brand_share_over_200k_km",
    "brand_automatic_share",
    "brand_petrol_share",
    "brand_diesel_share",
    "brand_ev_share",
    "brand_hybrid_share",
    "brand_turbo_share",
]

traficom_extended_features = [
    column for column in traficom_extended_candidates if column in train_df.columns
]

missing_traficom_extended_features = [
    column for column in traficom_extended_candidates if column not in train_df.columns
]

print("Found registry lifecycle features:")
print(registry_lifecycle_features)

print("\nMissing registry lifecycle features:")
print(missing_registry_lifecycle_features)

print("\nFound extended Traficom features:")
print(traficom_extended_features)

print("\nMissing extended Traficom features:")
print(missing_traficom_extended_features)

# Listing dynamics features describe scrape-window listing behavior,
# not registry lifecycle or vehicle registration-history features.
listing_dynamics_features = [
    "times_observed",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
]

# Keep this alias so the existing notebook cells continue to run.
lifecycle_features = listing_dynamics_features

Found registry lifecycle features:
['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span']

Missing registry lifecycle features:
[]

Found extended Traficom features:
['model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over_10y', 'brand_share_over_200k_km', 'brand_automatic_share', 'brand_petrol_share', 'brand_diesel_share', 'brand_ev_s

## 5. Select baseline feature set

In [7]:
features_baseline_only = baseline_features

assert len(features_baseline_only) > 0, "Add at least one column to baseline_features before training."

print("Number of baseline features:", len(features_baseline_only))
features_baseline_only

Number of baseline features: 13


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status']

## 6. Build X and y

In [8]:
missing_features = [column for column in features_baseline_only if column not in train_df.columns]
assert len(missing_features) == 0, f"These selected features are missing from the dataset: {missing_features}"

X_train = train_df[features_baseline_only].copy()
X_validation = validation_df[features_baseline_only].copy()

y_train = train_df["price"].copy()
y_validation = validation_df["price"].copy()

## 7. Log-transform target

In [9]:
y_train_log = np.log(y_train)

y_train_log.head()

0    5.179534
1    5.179534
2    5.179534
3    5.179534
4    3.165475
Name: price, dtype: float64

## 8. Detect column types

In [10]:
registry_lifecycle_candidates = [
    numeric_features := X_train.select_dtypes(include=["number", "bool"]).columns.tolist(),
    categorical_features := X_train.select_dtypes(include=["object", "category"]).columns.tolist()
]

numeric_features

['mileage', 'year_start', 'year_end', 'year_span', 'year_mid']

## 8. Detect column types

In [11]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [12]:
print("Numeric columns:", numeric_features)
print("\nCategorical columns:", categorical_features)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 9. Build preprocessing and model pipeline

In [13]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

In [14]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

## 10. Train baseline linear regression

In [15]:
baseline_model.fit(X_train, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 11. Predict and convert back to euro scale

In [16]:
validation_pred_log = baseline_model.predict(X_validation)

In [17]:
validation_pred = np.exp(validation_pred_log)

## 12. Validation matrics


In [18]:
validation_mae = mean_absolute_error(y_validation, validation_pred)
validation_rmse = np.sqrt(mean_squared_error(y_validation, validation_pred))

print("Validation MAE:", validation_mae)
print("Validation RMSE:", validation_rmse)

Validation MAE: 43.030238669492576
Validation RMSE: 155.90878235254524


## 13. Baseline + Traficom features

This section tests whether external Traficom enrichment improves prediction compared to the listing-only baseline.

In [19]:
features_baseline_plus_traficom = baseline_features + traficom_features

print("Number of baseline + Traficom features:", len(features_baseline_plus_traficom))
features_baseline_plus_traficom

Number of baseline + Traficom features: 26


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage']

## 14. Build X and y for baseline + Traficom

In [20]:
missing_traficom_features = [
    column for column in features_baseline_plus_traficom if column not in train_df.columns
]
assert len(missing_traficom_features) == 0, (
    f"These selected features are missing from the dataset: {missing_traficom_features}"
)

X_train_traficom = train_df[features_baseline_plus_traficom].copy()
X_validation_traficom = validation_df[features_baseline_plus_traficom].copy()

## 15. Detect column types again

In [21]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()

In [22]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 16. Build a fresh preprocessing and model pipeline

In [23]:
numeric_pipeline_traficom = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline_traficom = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

In [24]:
preprocessor_traficom = ColumnTransformer(transformers=[
    ("num", numeric_pipeline_traficom, numeric_features_traficom),
    ("cat", categorical_pipeline_traficom, categorical_features_traficom),
])

linear_regression_traficom = Pipeline(steps=[
    ("preprocessor", preprocessor_traficom),
    ("model", LinearRegression()),
])

## 17. Train baseline + Traficom model

In [25]:
linear_regression_traficom.fit(X_train_traficom, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 18. Predict on validation and convert back to euro scale

In [26]:
validation_pred_log_traficom = linear_regression_traficom.predict(X_validation_traficom)

In [27]:
validation_pred_traficom = np.exp(validation_pred_log_traficom)

## 19. Evaluate validation results

In [28]:
traficom_validation_mae = mean_absolute_error(y_validation, validation_pred_traficom)
traficom_validation_rmse = np.sqrt(mean_squared_error(y_validation, validation_pred_traficom))

In [29]:
print("Baseline + Traficom Validation MAE:", traficom_validation_mae)
print("Baseline + Traficom Validation RMSE:", traficom_validation_rmse)

Baseline + Traficom Validation MAE: 43.03214109394599
Baseline + Traficom Validation RMSE: 155.92138903426584


## 20. Baseline + Traficom + Registry lifecycle + Listing dynamics features

This section tests whether adding registry lifecycle features and listing dynamics features further improves prediction compared to the earlier models.

In [30]:
features_all = baseline_features + traficom_features + registry_lifecycle_features + listing_dynamics_features

print("Number of baseline + Traficom + registry lifecycle + listing dynamics features:", len(features_all))
features_all

Number of baseline + Traficom + registry lifecycle + listing dynamics features: 46


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage',
 'model_firstreg_total_2014_2026',
 'model_firstreg_recent_share',
 'model_firstreg_old_share',
 'model_firstreg_weighted_year',
 'model_firstreg_peak_year',
 'model_firstreg_peak_count',
 'model_firstreg_year_span',
 'brand_firstreg_total_2014_2026',
 'brand_firstreg_recent_share',
 'brand_firstreg_old_share',
 'brand_firstreg_weighted_year',
 'brand_firstreg_peak_year',
 'brand_firstreg_peak_count',
 'brand_firstreg_year_span',
 'times_observed',
 'observed_span_days',
 'p

## 21. Build X and y for all feature groups

In [31]:
missing_all_features = [column for column in features_all if column not in train_df.columns]
assert len(missing_all_features) == 0, (
    f"These selected features are missing from the dataset: {missing_all_features}"
)

X_train_all = train_df[features_all].copy()
X_validation_all = validation_df[features_all].copy()

## 22. Detect column types again

In [32]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()

In [33]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span', 'times_observed', 'observed_span_days', 'price_changed_flag', 'price_change_count', 'absolute_price_change', 'relative_price_change_pct']

Categorical columns: ['part_name',

## 23. Build a fresh preprocessing and model pipeline

In [34]:
numeric_pipeline_all = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline_all = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

In [35]:
preprocessor_all = ColumnTransformer(transformers=[
    ("num", numeric_pipeline_all, numeric_features_all),
    ("cat", categorical_pipeline_all, categorical_features_all),
])

linear_regression_all = Pipeline(steps=[
    ("preprocessor", preprocessor_all),
    ("model", LinearRegression()),
])

## 24. Train baseline + Traficom + registry lifecycle + listing dynamics model

In [36]:
linear_regression_all.fit(X_train_all, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 25. Predict on validation and convert back to euro scale

In [37]:
validation_pred_log_all = linear_regression_all.predict(X_validation_all)

In [38]:
validation_pred_all = np.exp(validation_pred_log_all)

## 26. Evaluate validation results

In [39]:
all_validation_mae = mean_absolute_error(y_validation, validation_pred_all)
all_validation_rmse = np.sqrt(mean_squared_error(y_validation, validation_pred_all))

In [40]:
print("Baseline + Traficom + Registry Lifecycle + Listing Dynamics Validation MAE:", all_validation_mae)
print("Baseline + Traficom + Registry Lifecycle + Listing Dynamics Validation RMSE:", all_validation_rmse)

Baseline + Traficom + Registry Lifecycle + Listing Dynamics Validation MAE: 39.49314913737193
Baseline + Traficom + Registry Lifecycle + Listing Dynamics Validation RMSE: 167.61712740610298


## 27. Compare all feature-set experiments

This section reruns the four main experiments in a compact way so the results are easy to compare.

In [41]:
excluded_features = {
    "product_id",
    "scrape_date",
    "brand_merge_key",
    "model_merge_key",
    "first_seen_date",
    "last_seen_date",
    "brand_is_known_model_family",
    "model_looks_like_part_taxonomy",
    "model_family_clean",
}

feature_sets = {
    "baseline only": baseline_features,
    "baseline + traficom_core": baseline_features + traficom_features,
    "baseline + traficom_core + registry_lifecycle": (
        baseline_features + traficom_features + registry_lifecycle_features
    ),
    "baseline + traficom_core + registry_lifecycle + traficom_extended": (
        baseline_features
        + traficom_features
        + registry_lifecycle_features
        + traficom_extended_features
    ),
    "baseline + traficom_core + registry_lifecycle + traficom_extended + listing_dynamics": (
        baseline_features
        + traficom_features
        + registry_lifecycle_features
        + traficom_extended_features
        + listing_dynamics_features
    ),
}

experiment_results = []

In [42]:
experiment_feature_details = []

for model_name, feature_list in feature_sets.items():
    requested_features = list(dict.fromkeys(feature_list))
    requested_features = [feature for feature in requested_features if feature not in excluded_features]

    available_features = [feature for feature in requested_features if feature in train_df.columns]
    missing_features = [feature for feature in requested_features if feature not in train_df.columns]

    X_train_current = train_df[available_features].copy()
    X_validation_current = validation_df[available_features].copy()

    numeric_features_current = X_train_current.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features_current = X_train_current.select_dtypes(include=["object", "category"]).columns.tolist()

    numeric_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor_current = ColumnTransformer(transformers=[
        ("num", numeric_pipeline_current, numeric_features_current),
        ("cat", categorical_pipeline_current, categorical_features_current),
    ])

    model_current = Pipeline(steps=[
        ("preprocessor", preprocessor_current),
        ("model", LinearRegression()),
    ])

    model_current.fit(X_train_current, y_train_log)

    validation_pred_log_current = model_current.predict(X_validation_current)
    validation_pred_current = np.exp(validation_pred_log_current)

    validation_mae_current = mean_absolute_error(y_validation, validation_pred_current)
    validation_rmse_current = np.sqrt(mean_squared_error(y_validation, validation_pred_current))

    print("=" * 70)
    print(f"Experiment: {model_name}")
    print(f"Raw columns requested: {len(requested_features)}")
    print(f"Raw columns actually used: {len(available_features)}")
    print(f"Missing candidate columns skipped: {len(missing_features)}")
    print(f"Validation MAE: {validation_mae_current:.4f}")
    print(f"Validation RMSE: {validation_rmse_current:.4f}")

    experiment_results.append({
        "experiment": model_name,
        "raw_columns_requested": len(requested_features),
        "raw_columns_used": len(available_features),
        "raw_columns_missing": len(missing_features),
        "validation_MAE": validation_mae_current,
        "validation_RMSE": validation_rmse_current,
    })

    experiment_feature_details.append({
        "experiment": model_name,
        "used_features_count": len(available_features),
        "used_features_list": available_features,
        "missing_features_list": missing_features,
    })

Experiment: baseline only
Raw columns requested: 13
Raw columns actually used: 13
Missing candidate columns skipped: 0
Validation MAE: 43.0302
Validation RMSE: 155.9088
Experiment: baseline + traficom_core
Raw columns requested: 26
Raw columns actually used: 26
Missing candidate columns skipped: 0
Validation MAE: 43.0321
Validation RMSE: 155.9214
Experiment: baseline + traficom_core + registry_lifecycle
Raw columns requested: 40
Raw columns actually used: 40
Missing candidate columns skipped: 0
Validation MAE: 43.0326
Validation RMSE: 155.9184
Experiment: baseline + traficom_core + registry_lifecycle + traficom_extended
Raw columns requested: 62
Raw columns actually used: 62
Missing candidate columns skipped: 0
Validation MAE: 43.0319
Validation RMSE: 155.9161
Experiment: baseline + traficom_core + registry_lifecycle + traficom_extended + listing_dynamics
Raw columns requested: 68
Raw columns actually used: 68
Missing candidate columns skipped: 0
Validation MAE: 39.4930
Validation RMSE